# A Laser Wakefield Accelerator

Follow [the wakefield lesson](https://blast-warpx.github.io/warpx-tutorials/a-laser-wakefield-accelerator.html) for setup, simulation, and interpretation. After the run finishes, execute the analysis cells below.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from openpmd_viewer import OpenPMDTimeSeries
from scipy.constants import e

In [ ]:
# Locate this notebook's folder even if the kernel started in its parent.
notebook_dir = Path.cwd()
if not (notebook_dir / "warpx_helpers.py").is_file():
    notebook_dir = next(
        (
            notebook_dir / name
            for name in ("laser-wakefield", "lwfa_warpx")
            if (notebook_dir / name / "warpx_helpers.py").is_file()
        ),
        notebook_dir,
    )
sys.path.insert(0, str(notebook_dir))
from warpx_helpers import plot_density_evolution, plot_snapshot  # isort: skip

out_folder = str(notebook_dir / "diags/diag1")
if not Path(out_folder).is_dir():
    raise FileNotFoundError(
        f"WarpX diagnostics not found at {out_folder}. Run the simulation first or update out_folder."
    )
print("Reading diagnostics from:", out_folder)

## Open the diagnostics

Choose a saved `iteration` and read electron positions (meters) and weights (physical electrons per macroparticle).


In [ ]:
series = OpenPMDTimeSeries(out_folder)
print("Saved steps:", series.iterations)
print("Species:", series.avail_species)
iteration = int(series.iterations[len(series.iterations) // 2])
z, x, w = series.get_particle(["z", "x", "w"], species="electrons", iteration=iteration)
print(f"Read {len(x):,} macroparticles at step {iteration}")

## Plot electron charge

Each bin shows charge magnitude in pC, integrated over y, for all saved electrons.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
hist = ax.hist2d(z * 1e6, x * 1e6, bins=80, weights=w * e * 1e12, cmap="magma")
fig.colorbar(hist[3], ax=ax, label="Charge per bin (pC)")
ax.set(xlabel="z (µm)", ylabel="x (µm)", title=f"Saved electrons · step {iteration}")
plt.show()

## Plot the transverse electric field

Read total `E_y` at the same step and slice through y = 0.


In [ ]:
laser, info = series.get_field("E", coord="y", iteration=iteration, slice_across="y")
# imshow needs x along the rows and z along the columns.
if info.axes[0] == "z":
    laser = laser.T

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
scale = max(abs(laser).max() / 1e9, 1e-12)
im = ax.imshow(
    laser / 1e9,
    origin="lower",
    aspect="auto",
    cmap="RdBu_r",
    extent=[info.z[0] * 1e6, info.z[-1] * 1e6, info.x[0] * 1e6, info.x[-1] * 1e6],
    vmin=-scale,
    vmax=scale,
)
fig.colorbar(im, ax=ax, label="Electric field Ey (GV/m)")
ax.set(xlabel="z (µm)", ylabel="x (µm)", title=f"Laser field, y = 0 · step {iteration}")
plt.show()

## Watch the wake evolve

Density slices share a color scale and use the moving coordinate `z-ct`.


In [ ]:
fig = plot_density_evolution(out_folder)
plt.show()

## Inspect a snapshot

Set `iteration` to a saved step and rerun the cell to plot density, accelerating field, electron spectrum, and energy histories.


In [ ]:
iteration = int(series.iterations[len(series.iterations) // 2])
fig = plot_snapshot(out_folder, iteration)
plt.show()